In [1]:
#Import of Libraries
import pandas as pd
import re
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
import sentencepiece
import google.protobuf

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from rouge_score import rouge_scorer
import random
import copy


/home/abdallah/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Reading the datasets and taking the two columns we need and renaming them to text/target

#Dataset_1-Training
SumArabic_tr = pd.read_json("sumarabic-1.0-train.jsonl" , lines = True)
SumArabic_tr = SumArabic_tr[["text" , "headline"]].rename(columns = {"headline" : "target"})

#Dataset_1-Testing
SumArabic_te = pd.read_json("sumarabic-1.0-test.jsonl" , lines = True)
SumArabic_te = SumArabic_te[["text" , "headline"]].rename(columns = {"headline" : "target"})


#Dataset_1-Validation
SumArabic_va = pd.read_json("sumarabic-1.0-valid.jsonl" , lines = True)
SumArabic_va = SumArabic_va[["text" , "headline"]].rename(columns = {"headline" : "target"})

#Dataset_2
AraSum = pd.read_csv("AraSum.csv" , sep = "\t" , engine = "python" , header = None , names = ["index" , "text" , "target"]) 
AraSum = AraSum.drop("index" , axis = 1)

#Dataset_3
EgyD = pd.read_parquet("train-00000-of-00001.parquet" , engine = "pyarrow")
EgyD = EgyD[["text" , "summarized_text"]].rename(columns = {"summarized_text" : "target"})

#Dataset_4
ArabicKagg = pd.read_csv("summarizdataset.csv")
ArabicKagg = ArabicKagg[["text" , "summarizer"]].rename(columns = {"summarizer" : "target"})




In [3]:
#Splitting the datasets into train/test

#First we concat the three unsplit datasets
df = pd.concat([AraSum , EgyD , ArabicKagg] , ignore_index=True)
df_train , df_test = train_test_split(df , test_size=0.15 , random_state=42)
df_train , df_valid = train_test_split(df_train , test_size = 0.15 , random_state = 42)

#We then add the already split dataset into the split
df_train = pd.concat([df_train , SumArabic_tr] , ignore_index=True)
df_test = pd.concat([df_test , SumArabic_te] , ignore_index=True)
df_valid = pd.concat([df_valid, SumArabic_va] , ignore_index=True)



In [4]:
#Function for Cleaning
def cleaning_words(x):
    x = re.sub(r' +', ' ', x)
    x = re.sub(r'"([^"]+)"', r'\1', x, flags=re.UNICODE)
    x = re.sub(r'[»«]', '', x)
    x = re.sub(r'[a-zA-Z]', '', x)
    x = re.sub(r'\(\d+\s*/\s*\w+\)', '', x) 
    x = re.sub(r'[()/]', '', x)
    x = re.sub(r'[ء-ي]\.[ء-ي]', '', x)
    x = re.sub(r'\s[ء-ي]\s', ' ', x)
    x = re.sub(r'\n', '', x)
    x = x.replace('\xa0', ' ')
    x = x.replace('\u200f', '')
    x = x.replace('\u200e', '')
    x = x.replace('\ufeff', '')
    x = x.replace('\u200c', '')
    x = x.replace('\u200b', '')
    x = x.replace('\u202c', '')
    x = x.replace('\u202a', '')
    x = x.replace('\u202b', '')
    x = x.replace('…', '.')
    x = x.replace('٪', '%')
    x = x.replace('٬', '')
    x = x.replace('―', '-')
    return x

In [5]:
#Calling the function to clean text and target
df_train["processed_data"] = df_train["text"].map(cleaning_words)
df_test["processed_data"] = df_test["text"].map(cleaning_words)
df_valid["processed_data"] = df_valid["text"].map(cleaning_words)

df_train["processed_target"] = df_train["target"].map(cleaning_words)
df_test["processed_target"] = df_test["target"].map(cleaning_words)
df_valid["processed_target"] = df_valid["target"].map(cleaning_words)



In [6]:
#Function for Tokenization for both text and target
tokenizer = AutoTokenizer.from_pretrained('moussaKam/AraBART')

def tokenize(batch):
    inputs = tokenizer(
        batch['processed_data'],
        max_length=1024,
        truncation=True,
        padding='max_length'
    )
    targets = tokenizer(
        batch['processed_target'],
        max_length=256,
        truncation=True,
        padding='max_length'
    )
    inputs['labels'] = targets['input_ids']
    return inputs

In [7]:
#Calling the function to tokenize the data (same function does text and target)
df_train = df_train.apply(tokenize , axis = 1)
df_test = df_test.apply(tokenize , axis = 1)
df_valid = df_valid.apply(tokenize , axis = 1)

In [8]:
#Making it into a df after the apply axis = 1
df_train = pd.DataFrame(df_train.tolist())
df_valid = pd.DataFrame(df_valid.tolist())
df_test = pd.DataFrame(df_test.tolist())



In [9]:
# Initialize the lists for training pair and validation pairs
train_pairs = []
valid_pairs = []

MAX_LEN_SRC = 256
MAX_LEN_TGT = 64

for i in range(len(df_train)):
    row = df_train.iloc[i]
    
    # Using tokenized input_ids from the preproccesing phase
    enc = row['input_ids'][:MAX_LEN_SRC]
    mask = row['attention_mask'][:MAX_LEN_SRC]
    dec = row['labels'][:MAX_LEN_TGT]

    # Padding
    enc = enc + [tokenizer.pad_token_id] * (MAX_LEN_SRC - len(enc))
    mask = mask + [0] * (MAX_LEN_SRC - len(mask))
    dec = dec + [tokenizer.pad_token_id] * (MAX_LEN_TGT - len(dec))

    decoder_input = dec[:-1]
    decoder_target = dec[1:]

    train_pairs.append((enc, mask, decoder_input, decoder_target))

#Same as training but for val
for i in range(len(df_valid)):
    row = df_valid.iloc[i]
    enc = row['input_ids'][:MAX_LEN_SRC]
    mask = row['attention_mask'][:MAX_LEN_SRC]
    dec = row['labels'][:MAX_LEN_TGT]

    enc = enc + [tokenizer.pad_token_id] * (MAX_LEN_SRC - len(enc))
    mask = mask + [0] * (MAX_LEN_SRC - len(mask))
    dec = dec + [tokenizer.pad_token_id] * (MAX_LEN_TGT - len(dec))

    decoder_input = dec[:-1]
    decoder_target = dec[1:]

    valid_pairs.append((enc, mask, decoder_input, decoder_target))

In [10]:

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 32
EMBEDDING_DIM = 128
HIDDEN_SIZE = 256
NUM_EPOCHS = 10

print(f"Current Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


Current Device: NVIDIA GeForce RTX 4060 Laptop GPU


In [11]:
#Creating the datasets
train_data = TensorDataset(
    torch.tensor([p[0] for p in train_pairs]), 
    torch.tensor([p[1] for p in train_pairs]), 
    torch.tensor([p[2] for p in train_pairs]), 
    torch.tensor([p[3] for p in train_pairs])  
)

valid_data = TensorDataset(
    torch.tensor([p[0] for p in valid_pairs]),
    torch.tensor([p[1] for p in valid_pairs]),
    torch.tensor([p[2] for p in valid_pairs]),
    torch.tensor([p[3] for p in valid_pairs])
)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE)

In [12]:
#Embeddings and initializing weights
VOCAB_SIZE = tokenizer.vocab_size
PAD_ID = tokenizer.pad_token_id

encoder_embedding = nn.Embedding(VOCAB_SIZE, EMBEDDING_DIM).to(DEVICE)
encoder_lstm = nn.LSTM(EMBEDDING_DIM, HIDDEN_SIZE, batch_first=True, bidirectional=True).to(DEVICE)

decoder_embedding = nn.Embedding(VOCAB_SIZE, EMBEDDING_DIM).to(DEVICE)
decoder_lstm = nn.LSTM(EMBEDDING_DIM + HIDDEN_SIZE*2, HIDDEN_SIZE, batch_first=True).to(DEVICE)

W1 = nn.Linear(HIDDEN_SIZE*2, HIDDEN_SIZE).to(DEVICE)
W2 = nn.Linear(HIDDEN_SIZE, HIDDEN_SIZE).to(DEVICE)
V  = nn.Linear(HIDDEN_SIZE, 1).to(DEVICE)

fc = nn.Linear(HIDDEN_SIZE, VOCAB_SIZE).to(DEVICE)


In [13]:

params = list(encoder_embedding.parameters()) +          list(encoder_lstm.parameters()) +          list(decoder_embedding.parameters()) +          list(decoder_lstm.parameters()) +          list(W1.parameters()) + list(W2.parameters()) + list(V.parameters()) +          list(fc.parameters())

optimizer = torch.optim.Adam(params, lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

#Early stopping values
PATIENCE = 3
MIN_DELTA = 0.0

best_val_loss = float("inf")
epochs_without_improvement = 0
best_model_state = None


In [14]:
#The actual training for the Model
for epoch in range(NUM_EPOCHS):
    total_loss = 0
    for batch in train_loader:
        encoder_ids = batch[0].to(DEVICE)
        src_mask = batch[1].to(DEVICE)     
        decoder_input = batch[2].to(DEVICE)
        decoder_target = batch[3].to(DEVICE)

        optimizer.zero_grad()

        #Encoder
        enc_emb = encoder_embedding(encoder_ids)
        encoder_outputs, (h, c) = encoder_lstm(enc_emb)

        #iinitialize the Decoder's hidden state
        h = torch.cat((h[0], h[1]), dim=1).unsqueeze(0).contiguous() 
        c = torch.cat((c[0], c[1]), dim=1).unsqueeze(0).contiguous() 
        decoder_hidden = (h[:, :, :HIDDEN_SIZE].contiguous(), c[:, :, :HIDDEN_SIZE].contiguous())

        batch_size, tgt_len = decoder_input.shape
        outputs = torch.zeros(batch_size, tgt_len, VOCAB_SIZE).to(DEVICE)

        W1_out = W1(encoder_outputs)

        for t in range(tgt_len):
            dec_emb = decoder_embedding(decoder_input[:, t].unsqueeze(1))
            hidden_state = decoder_hidden[0].squeeze(0)

            #Solving an getting tthe value for Attention Score
            score = V(torch.tanh(W1_out + W2(hidden_state).unsqueeze(1)))

            #Masking
            score = score.masked_fill(src_mask.unsqueeze(2) == 0, -1e9)

            #context vector / softmax
            attention_weights = torch.softmax(score, dim=1)
            context = torch.sum(attention_weights * encoder_outputs, dim=1).unsqueeze(1)

            #Decoder
            lstm_input = torch.cat((dec_emb, context), dim=2)

            h_state, c_state = decoder_hidden
            decoder_hidden = (h_state.contiguous(), c_state.contiguous())

            dec_output, decoder_hidden = decoder_lstm(lstm_input, decoder_hidden)
            outputs[:, t, :] = fc(dec_output.squeeze(1))

        #Loss Function
        loss = criterion(outputs.view(-1, VOCAB_SIZE), decoder_target.reshape(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)

    #Validation Loss
    val_loss = 0
    with torch.no_grad():
        for batch in valid_loader:
            encoder_ids = batch[0].to(DEVICE)
            src_mask = batch[1].to(DEVICE)     
            decoder_input = batch[2].to(DEVICE)
            decoder_target = batch[3].to(DEVICE)

            #Encoder
            enc_emb = encoder_embedding(encoder_ids)
            encoder_outputs, (h, c) = encoder_lstm(enc_emb)

            #iinitialize the Decoder's hidden state
            h = torch.cat((h[0], h[1]), dim=1).unsqueeze(0).contiguous() 
            c = torch.cat((c[0], c[1]), dim=1).unsqueeze(0).contiguous() 
            decoder_hidden = (h[:, :, :HIDDEN_SIZE].contiguous(), c[:, :, :HIDDEN_SIZE].contiguous())

            batch_size, tgt_len = decoder_input.shape
            outputs = torch.zeros(batch_size, tgt_len, VOCAB_SIZE).to(DEVICE)

            W1_out = W1(encoder_outputs)

            for t in range(tgt_len):
                dec_emb = decoder_embedding(decoder_input[:, t].unsqueeze(1))
                hidden_state = decoder_hidden[0].squeeze(0)

                #Solving an getting tthe value for Attention Score
                score = V(torch.tanh(W1_out + W2(hidden_state).unsqueeze(1)))

                #Masking
                score = score.masked_fill(src_mask.unsqueeze(2) == 0, -1e9)

                #context vector / softmax
                attention_weights = torch.softmax(score, dim=1)
                context = torch.sum(attention_weights * encoder_outputs, dim=1).unsqueeze(1)

                #Decoder
                lstm_input = torch.cat((dec_emb, context), dim=2)

                h_state, c_state = decoder_hidden
                decoder_hidden = (h_state.contiguous(), c_state.contiguous())

                dec_output, decoder_hidden = decoder_lstm(lstm_input, decoder_hidden)
                outputs[:, t, :] = fc(dec_output.squeeze(1))

            #Loss Function
            loss = criterion(outputs.view(-1, VOCAB_SIZE), decoder_target.reshape(-1))
            val_loss += loss.item()

    val_loss = val_loss / len(valid_loader)

    print(f"Epoch {epoch+1}, Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

    #Early stopping
    if val_loss < best_val_loss - MIN_DELTA:
        best_val_loss = val_loss
        epochs_without_improvement = 0

        best_model_state = {
            'encoder_embedding': copy.deepcopy(encoder_embedding.state_dict()),
            'encoder_lstm': copy.deepcopy(encoder_lstm.state_dict()),
            'decoder_embedding': copy.deepcopy(decoder_embedding.state_dict()),
            'decoder_lstm': copy.deepcopy(decoder_lstm.state_dict()),
            'W1': copy.deepcopy(W1.state_dict()),
            'W2': copy.deepcopy(W2.state_dict()),
            'V': copy.deepcopy(V.state_dict()),
            'fc': copy.deepcopy(fc.state_dict())
        }

    else:
        epochs_without_improvement += 1
        print(f"Early stopping counter: {epochs_without_improvement}/{PATIENCE}")

        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

OutOfMemoryError: CUDA out of memory. Tried to allocate 386.00 MiB. GPU 0 has a total capacity of 7.62 GiB of which 340.06 MiB is free. Process 8074 has 6.94 GiB memory in use. Including non-PyTorch memory, this process has 332.00 MiB memory in use. Of the allocated memory 205.60 MiB is allocated by PyTorch, and 16.40 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:

#Loading the best model before saving
if best_model_state is not None:
    encoder_embedding.load_state_dict(best_model_state['encoder_embedding'])
    encoder_lstm.load_state_dict(best_model_state['encoder_lstm'])
    decoder_embedding.load_state_dict(best_model_state['decoder_embedding'])
    decoder_lstm.load_state_dict(best_model_state['decoder_lstm'])
    W1.load_state_dict(best_model_state['W1'])
    W2.load_state_dict(best_model_state['W2'])
    V.load_state_dict(best_model_state['V'])
    fc.load_state_dict(best_model_state['fc'])

torch.save({
    'encoder_embedding': encoder_embedding.state_dict(),
    'encoder_lstm': encoder_lstm.state_dict(),
    'decoder_embedding': decoder_embedding.state_dict(),
    'decoder_lstm': decoder_lstm.state_dict(),
    'W1': W1.state_dict(),
    'W2': W2.state_dict(),
    'V': V.state_dict(),
    'fc': fc.state_dict()
}, "seq2seq_model.pth")

print("Model saved!")


In [ ]:
scorer = rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=True)

samples = random.sample(valid_pairs, min(50, len(valid_pairs)))

scores = []

for enc, mask, decoder_input, decoder_target in samples:
    tgt = decoder_target

    pred = tgt[:len(tgt)//2]  # dummy prediction

    pred_text = " ".join(map(str, pred))
    tgt_text = " ".join(map(str, tgt))

    score = scorer.score(tgt_text, pred_text)
    scores.append(score['rouge1'].fmeasure)

print("ROUGE-1:", sum(scores)/len(scores))